# TabM exp_045 fold0 스크린 — native cross 임베딩 (#027 NN판)

realmlp_fe_v2에서 Driver=TE 유지, Race_Compound·Race_Year를 OOF-TE→native 임베딩으로 분기.  
목적: RealMLP(전부 TE)와 표현 분기 → corr↓. max_folds=1.  
비교: exp_038(no-bins TE) fold0 **0.951988**, TabM↔RealMLP fold0 corr **0.9676**.

In [ ]:
# 1) input 자동탐색 (마운트 비표준) — torch import 前
import sys, os, glob, subprocess
from pathlib import Path
print('/kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')
c = glob.glob('/kaggle/input/**/src/config.py', recursive=True); assert c, 'src/config.py 못 찾음'
SRC_ROOT = str(Path(c[0]).parents[1]); print('SRC_ROOT:', SRC_ROOT)
cc = glob.glob('/kaggle/input/**/playground-series-s6e5', recursive=True); assert cc, '대회 폴더 못 찾음'
COMP = Path(cc[0]); print('COMP:', COMP)
ac = glob.glob('/kaggle/input/**/f1_strategy_dataset*.csv', recursive=True); assert ac, '증강 csv 못 찾음'
AUG = Path(ac[0]); print('AUG:', AUG)

In [ ]:
# 2) GPU 종류 감지 → 조건부 torch 재설치, 그 위에 프로젝트 deps
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU)
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
if 'P100' in GPU:
    print('P100(sm_60) → cu121 torch 재설치')
    pip('torch==2.5.1','torchvision==0.20.1','torchaudio==2.5.1',
        '--index-url','https://download.pytorch.org/whl/cu121')
else:
    print('T4 등(sm_75+) → Kaggle 기본 torch 유지')
pip('pytabkit','hydra-core','python-dotenv')

In [ ]:
# 3) torch CUDA 실연산 검증 + import 체인 fast-fail
import torch
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0))
_x = torch.randn(256, 256, device='cuda'); _v = (_x @ _x).sum().item()
print('CUDA matmul OK')
sys.path.insert(0, SRC_ROOT)
from src import config
from src.train_tabm import run
print('import OK:', config.__file__)

In [ ]:
# 4) 경로 override
config.TRAIN_PATH = COMP / 'train.csv'
config.TEST_PATH = COMP / 'test.csv'
config.SAMPLE_SUBMISSION_PATH = COMP / 'sample_submission.csv'
config.SOURCE_AUG_PATH = AUG
out = Path('/kaggle/working')
config.OOF_DIR = out / 'oof'; config.SUBMISSION_DIR = out / 'submissions'; config.LOG_DIR = out / 'logs'
import pandas as pd
_a = pd.read_csv(config.SOURCE_AUG_PATH); print('AUG shape:', _a.shape)
assert len(_a) == 101371, f'증강 행수 불일치: {len(_a)}'
assert config.TRAIN_PATH.exists(), f'train.csv 없음: {config.TRAIN_PATH}'

In [ ]:
# 5) cfg 구성 — tabm_natcross (Driver TE 유지, cross 2개 native), max_folds=1
from omegaconf import OmegaConf
CONF = Path(SRC_ROOT) / 'conf'
cfg = OmegaConf.create({
    'exp_id': 'exp_045_tabm_natcross_fold0',
    'notes': 'TabM fold0 screen: native cross embed(Race_Compound/Race_Year), Driver TE 유지',
    'use_wandb': False,
    'max_folds': 1,
    'model': OmegaConf.load(CONF / 'model' / 'tabm.yaml'),
    'features': OmegaConf.load(CONF / 'features' / 'tabm_natcross.yaml'),
    'augment': {'enabled': True, 'weight': 1.0},
})
print(OmegaConf.to_yaml(cfg))


In [ ]:
# 6) 학습 (fold0만). TabM_D 기본 스케줄 → 예상 ~15-25분
import time
t0 = time.time()
result = run(cfg)
print(result, f'\n총 {time.time()-t0:.0f}s')


In [ ]:
# 7) fold0 AUC + RealMLP corr (decorrelation 판정용)
f0 = result.get('fold_scores', [None])[0]
print('fold0 AUC =', f0, '| exp_038(no-bins TE) fold0 = 0.951988')
# fold0 OOF로 RealMLP(exp_032) corr 측정 — RealMLP OOF가 input에 있으면
oof_path = out / 'oof' / 'exp_045_tabm_natcross_fold0.csv'
if oof_path.exists():
    print('OOF csv:', pd.read_csv(oof_path).shape)
log_path = out / 'logs' / 'exp_045_tabm_natcross_fold0.json'
if log_path.exists():
    import json
    with open(log_path) as f:
        print('LOG:', json.dumps(json.load(f), indent=2, default=str))
